# CIFAR-10 Project 1: knowledge distillation (Run D)

Trains one new model under 5M parameters to copy the averaged predictions of Runs A, B and C.
All three teachers were trained from scratch on the permitted data.
Only the new student model makes the Kaggle predictions.

**To run:** use `c12m85-a100-1` with 1 GPU and at least 3 hours left.
Then choose **Kernel → Restart Kernel and Run All Cells**.

**Before relying on it:** ask course staff whether a three-model teacher is allowed.

**If the session dies**, relaunch and run all cells again. It resumes from its last checkpoint.


## 1. Configuration

In [1]:
import copy, csv, json, os, random, subprocess, sys, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path(os.environ.get('CIFAR_ROOT', Path.home() / 'cifar10-project1'))
ROOT.mkdir(parents=True, exist_ok=True)
SMOKE = os.environ.get('CIFAR_SMOKE') == '1'   # short test run only

CFG = dict(
    seed=7,   # different from the teachers
    distill_alpha=0.7, distill_temperature=4.0,
    # Model: pre-activation wide ResNet, 3 stages of 2-convolution residual blocks
    widths=(59, 118, 236), blocks=(4, 4, 4),
    # Optimization
    epochs=300, warmup_epochs=5, batch_size=128, lr=0.1,
    momentum=0.9, weight_decay=5e-4, label_smoothing=0.1,
    # Augmentation
    random_erasing_p=0.25, mix_prob=0.5, mixup_alpha=0.2, cutmix_alpha=1.0,
    mix_off_last_epochs=15,
    # Averaged weights, used for all evaluation and prediction
    ema_decay=0.999,
    checkpoint_every=5,
)
if SMOKE:
    CFG.update(epochs=2, warmup_epochs=1, batch_size=64, mix_off_last_epochs=1, checkpoint_every=1)
print(json.dumps(CFG, indent=1))

{
 "seed": 7,
 "distill_alpha": 0.7,
 "distill_temperature": 4.0,
 "widths": [
  59,
  118,
  236
 ],
 "blocks": [
  4,
  4,
  4
 ],
 "epochs": 300,
 "warmup_epochs": 5,
 "batch_size": 128,
 "lr": 0.1,
 "momentum": 0.9,
 "weight_decay": 0.0005,
 "label_smoothing": 0.1,
 "random_erasing_p": 0.25,
 "mix_prob": 0.5,
 "mixup_alpha": 0.2,
 "cutmix_alpha": 1.0,
 "mix_off_last_epochs": 15,
 "ema_decay": 0.999,
 "checkpoint_every": 5
}


## 2. Environment check

In [2]:
import numpy as np
import torch
import torchvision
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

print('Python  ', sys.version.split()[0])
print('Torch   ', torch.__version__, '| torchvision', torchvision.__version__)
print('CUDA    ', torch.cuda.is_available())
if not SMOKE:
    assert torch.cuda.is_available(), 'No GPU. Launch the session on a GPU partition.'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    print('GPU     ', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
try:
    CPUS = len(os.sched_getaffinity(0))
except AttributeError:
    CPUS = os.cpu_count() or 2
WORKERS = 0 if SMOKE else max(2, min(10, CPUS - 2))
print('CPUs    ', CPUS, '| data workers', WORKERS)

Python   3.9.18
Torch    2.8.0+cu128 | torchvision 0.23.0+cu128
CUDA     True
GPU      NVIDIA A100-SXM4-40GB
CPUs     12 | data workers 10


## 3. Load the competition data

Finds the data folder, or unpacks `data.zip` from your home folder, then caches all images in one file.
Labels come only from `train_labels.csv`.

In [3]:
def find_data_dir():
    searched = [ROOT / 'kaggle_data', Path.home() / 'cifar10-resnet' / 'kaggle_data']
    for folder in searched:
        hits = list(folder.rglob('train_labels.csv')) if folder.is_dir() else []
        if hits:
            return hits[0].parent
    for folder in (Path.cwd(), Path.home(), ROOT):
        for archive in sorted(folder.glob('data*.zip')):
            target = ROOT / 'kaggle_data'
            print('Unpacking', archive)
            with zipfile.ZipFile(archive) as z:
                z.extractall(target)
            return next(target.rglob('train_labels.csv')).parent
    raise FileNotFoundError('Upload the course data.zip to your home folder and rerun this cell.')

def read_rows(path):
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

def read_png(path):
    with Image.open(path) as image:
        return np.asarray(image.convert('RGB'), dtype=np.uint8)

def read_folder(folder, names):
    with ThreadPoolExecutor(8) as pool:
        return np.stack(list(pool.map(read_png, [folder / name for name in names])))

DATA_DIR = find_data_dir()
train_rows = read_rows(DATA_DIR / 'train_labels.csv')
test_rows = read_rows(DATA_DIR / 'test.csv')
sample_rows = read_rows(DATA_DIR / 'sample_submission.csv')
assert list(sample_rows[0]) == ['id', 'label']
test_file_by_id = {row['id']: row['filename'] for row in test_rows}
SAMPLE_IDS = [row['id'] for row in sample_rows]
assert set(SAMPLE_IDS) == set(test_file_by_id), 'test.csv and sample_submission.csv IDs differ'

cache = ROOT / 'cache.npz'
if cache.exists():
    stored = np.load(cache)
    X_train, Y_train, X_test = stored['x_train'], stored['y_train'], stored['x_test']
else:
    start = time.time()
    X_train = read_folder(DATA_DIR / 'train_images', [row['filename'] for row in train_rows])
    Y_train = np.array([int(row['label']) for row in train_rows], dtype=np.int64)
    # Test images are stored in sample_submission order, so predictions line up
    X_test = read_folder(DATA_DIR / 'test_images', [test_file_by_id[i] for i in SAMPLE_IDS])
    np.savez(cache, x_train=X_train, y_train=Y_train, x_test=X_test)
    print(f'Cached images in {time.time() - start:.0f}s')

assert X_train.shape == (50000, 32, 32, 3) and X_test.shape == (2000, 32, 32, 3)
assert Y_train.min() == 0 and Y_train.max() == 9
assert np.bincount(Y_train).tolist() == [5000] * 10, 'Expected 5,000 images per class'

# Fixed stratified 10% validation split, used by every experiment
split_rng = np.random.default_rng(CFG['seed'])
val_parts, train_parts = [], []
for label in range(10):
    members = split_rng.permutation(np.flatnonzero(Y_train == label))
    val_parts.append(members[:500])
    train_parts.append(members[500:])
TRAIN_IDX, VAL_IDX = np.sort(np.concatenate(train_parts)), np.sort(np.concatenate(val_parts))
assert not set(TRAIN_IDX.tolist()) & set(VAL_IDX.tolist())
if SMOKE:
    TRAIN_IDX, VAL_IDX = TRAIN_IDX[:512], VAL_IDX[:256]
print('Train', len(TRAIN_IDX), '| validation', len(VAL_IDX), '| test', len(X_test))

Train 45000 | validation 5000 | test 2000


## 4. Augmentation

In [4]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=CFG['random_erasing_p']),
])

class TrainImages(Dataset):
    def __init__(self, images, labels):
        self.images, self.labels = images, labels
    def __len__(self):
        return len(self.images)
    def __getitem__(self, index):
        return train_transform(Image.fromarray(self.images[index])), int(self.labels[index])

def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)

def mix_batch(x, y):
    """Applies mixup or CutMix on the GPU. Returns inputs, both targets, and the mix weight."""
    order = torch.randperm(x.size(0), device=x.device)
    if random.random() < 0.5:
        lam = float(np.random.beta(CFG['mixup_alpha'], CFG['mixup_alpha']))
        return lam * x + (1 - lam) * x[order], y, y[order], lam
    lam = float(np.random.beta(CFG['cutmix_alpha'], CFG['cutmix_alpha']))
    height, width = x.shape[2:]
    cut = (1 - lam) ** 0.5
    box_h, box_w = int(height * cut), int(width * cut)
    cy, cx = random.randrange(height), random.randrange(width)
    y1, y2 = max(cy - box_h // 2, 0), min(cy + box_h // 2, height)
    x1, x2 = max(cx - box_w // 2, 0), min(cx + box_w // 2, width)
    x = x.clone()
    x[:, :, y1:y2, x1:x2] = x[order][:, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (height * width)
    return x, y, y[order], lam

## 5. Model

Pre-activation wide ResNet: every block adds a learned residual `F(x)` to a skip connection `S(x)`.
The assertion stops the notebook if the trainable parameter count exceeds 5,000,000.

In [5]:
class PreActBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.norm1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.norm2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.project = None
        if stride != 1 or in_channels != out_channels:
            self.project = nn.Conv2d(in_channels, out_channels, 1, stride, bias=False)

    def forward(self, x):
        activated = F.relu(self.norm1(x), inplace=True)
        skip = x if self.project is None else self.project(activated)
        out = self.conv1(activated)
        out = self.conv2(F.relu(self.norm2(out), inplace=True))
        return out + skip

class EfficientResNet(nn.Module):
    def __init__(self, widths, blocks, classes=10):
        super().__init__()
        self.stem = nn.Conv2d(3, widths[0] // 2, 3, 1, 1, bias=False)
        layers, previous = [], widths[0] // 2
        for stage, (width, depth) in enumerate(zip(widths, blocks)):
            for index in range(depth):
                layers.append(PreActBlock(previous, width, 2 if stage > 0 and index == 0 else 1))
                previous = width
        self.blocks = nn.Sequential(*layers)
        self.norm = nn.BatchNorm2d(previous)
        self.head = nn.Linear(previous, classes)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        x = F.relu(self.norm(self.blocks(self.stem(x))), inplace=True)
        return self.head(F.adaptive_avg_pool2d(x, 1).flatten(1))

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

PARAMETERS = count_trainable(EfficientResNet(CFG['widths'], CFG['blocks']))
print(f'TRAINABLE PARAMETERS: {PARAMETERS:,}')
assert PARAMETERS <= 5_000_000, 'Model exceeds the competition limit'

TRAINABLE PARAMETERS: 4,980,392


## Load the teacher models

In [6]:
TEACHER_RUNS = ['validation_run', 'all_data_run', 'robust_all_data']
TEACHERS = []
for run in TEACHER_RUNS:
    path = ROOT / 'results' / run / 'final_model.pt'
    assert path.exists(), f'Missing teacher: {path}'
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    teacher = EfficientResNet(state['widths'], state['blocks']).to(DEVICE).eval()
    teacher.load_state_dict(state['model'])
    teacher.requires_grad_(False)
    TEACHERS.append(teacher)
print('Teachers loaded:', len(TEACHERS))


Teachers loaded: 3


## 6. Evaluation and prediction

In [7]:
_mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
_std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
AMP = dict(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == 'cuda')

@torch.inference_mode()
def class_probabilities(net, images, flip_average=True, batch_size=500):
    net.eval()
    outputs = []
    for start in range(0, len(images), batch_size):
        batch = torch.from_numpy(images[start:start + batch_size]).to(DEVICE)
        batch = (batch.permute(0, 3, 1, 2).float() / 255 - _mean) / _std
        with torch.autocast(**AMP):
            probs = net(batch).softmax(1)
            if flip_average:
                probs = (probs + net(batch.flip(3)).softmax(1)) / 2
        outputs.append(probs.float().cpu())
    return torch.cat(outputs).numpy()

def accuracy(net, images, labels, flip_average=False):
    return float((class_probabilities(net, images, flip_average).argmax(1) == labels).mean())

def write_submission(net, path):
    labels = class_probabilities(net, X_test).argmax(1)
    assert len(labels) == len(SAMPLE_IDS) == 2000 and 0 <= labels.min() and labels.max() <= 9
    with open(path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'label'])
        writer.writerows(zip(SAMPLE_IDS, labels.tolist()))
    rows = read_rows(path)
    assert [row['id'] for row in rows] == SAMPLE_IDS, 'Submission order differs from the sample'
    return np.bincount(labels, minlength=10).tolist()

## 7. Training loop

Resumes from `last.pt`. Results are close but not bit-identical between runs, because GPU operations
and data workers are not fully deterministic. Record that in the report.

In [8]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def learning_rate(step, steps_per_epoch):
    warmup = CFG['warmup_epochs'] * steps_per_epoch
    total = CFG['epochs'] * steps_per_epoch
    if step < warmup:
        return CFG['lr'] * (step + 1) / warmup
    return CFG['lr'] * 0.5 * (1 + np.cos(np.pi * (step - warmup) / (total - warmup)))

def parameter_groups(model):
    # Weight decay applies to convolution and linear weights, not norms or biases
    decay = [p for p in model.parameters() if p.ndim > 1]
    no_decay = [p for p in model.parameters() if p.ndim <= 1]
    return [dict(params=decay, weight_decay=CFG['weight_decay']), dict(params=no_decay, weight_decay=0.0)]

def save_atomic(state, path):
    temporary = path.with_suffix('.tmp')
    torch.save(state, temporary)
    os.replace(temporary, path)

def run_experiment(name, indices, use_validation):
    out = ROOT / 'results' / name
    out.mkdir(parents=True, exist_ok=True)
    log_path, history_path, last_path = out / 'train.log', out / 'history.csv', out / 'last.pt'
    def log(message):
        print(message, flush=True)
        with open(log_path, 'a') as f:
            f.write(message + '\n')

    set_seed(CFG['seed'])
    model = EfficientResNet(CFG['widths'], CFG['blocks']).to(DEVICE)
    assert count_trainable(model) == PARAMETERS <= 5_000_000
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(CFG['ema_decay']), use_buffers=True)
    optimizer = torch.optim.SGD(parameter_groups(model), lr=CFG['lr'],
                                momentum=CFG['momentum'], nesterov=True)
    (out / 'config.json').write_text(json.dumps(dict(CFG, run=name, parameters=PARAMETERS,
        train_images=len(indices), torch=torch.__version__), indent=2))

    first_epoch, history = 1, []
    if last_path.exists():
        state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state['model'])
        ema.load_state_dict(state['ema'])
        optimizer.load_state_dict(state['optimizer'])
        first_epoch, history = state['epoch'] + 1, state['history']
        log(f'Resumed {name} after epoch {state["epoch"]}')

    loader = DataLoader(TrainImages(X_train[indices], Y_train[indices]),
                        batch_size=CFG['batch_size'], shuffle=True, drop_last=True,
                        num_workers=WORKERS, worker_init_fn=seed_worker,
                        pin_memory=DEVICE.type == 'cuda', persistent_workers=WORKERS > 0)
    steps = len(loader)
    set_seed(CFG['seed'] + first_epoch)   # different augmentations after a resume
    run_start = time.time()

    for epoch in range(first_epoch, CFG['epochs'] + 1):
        epoch_start = time.time()
        model.train()
        loss_sum, seen = 0.0, 0
        mixing_on = epoch <= CFG['epochs'] - CFG['mix_off_last_epochs']
        for batch_index, (x, y) in enumerate(loader):
            lr = learning_rate((epoch - 1) * steps + batch_index, steps)
            for group in optimizer.param_groups:
                group['lr'] = lr
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            y_other, lam = y, 1.0
            if mixing_on and random.random() < CFG['mix_prob']:
                x, y, y_other, lam = mix_batch(x, y)
            with torch.no_grad(), torch.autocast(**AMP):
                teacher_probs = sum((teacher(x) / CFG['distill_temperature']).softmax(1)
                                    for teacher in TEACHERS).float() / len(TEACHERS)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(**AMP):
                logits = model(x)
                loss = (lam * F.cross_entropy(logits, y, label_smoothing=CFG['label_smoothing'])
                        + (1 - lam) * F.cross_entropy(logits, y_other, label_smoothing=CFG['label_smoothing']))
            T, alpha = CFG['distill_temperature'], CFG['distill_alpha']
            soft_loss = F.kl_div(F.log_softmax(logits.float() / T, 1), teacher_probs,
                                 reduction='batchmean') * T * T
            loss = (1 - alpha) * loss + alpha * soft_loss
            loss.backward()
            optimizer.step()
            ema.update_parameters(model)
            loss_sum += loss.item() * y.size(0)
            seen += y.size(0)

        assert np.isfinite(loss_sum), 'Training loss became NaN or infinite'
        record = dict(epoch=epoch, lr=lr, train_loss=loss_sum / seen,
                      seconds=round(time.time() - epoch_start, 1))
        if use_validation:
            record['val_acc_ema'] = accuracy(ema, X_train[VAL_IDX], Y_train[VAL_IDX])
        history.append(record)
        with open(history_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(record))
            writer.writeheader()
            writer.writerows(history)
        if epoch % CFG['checkpoint_every'] == 0 or epoch == CFG['epochs']:
            save_atomic(dict(model=model.state_dict(), ema=ema.state_dict(),
                             optimizer=optimizer.state_dict(), epoch=epoch, history=history), last_path)
        if epoch - first_epoch == 1:
            per_epoch = (time.time() - run_start) / 2
            log(f'Estimate: {per_epoch:.0f}s per epoch, {per_epoch * (CFG["epochs"] - epoch) / 3600:.1f}h left in {name}')
        if epoch % 5 == 0 or epoch == CFG['epochs'] or epoch == first_epoch:
            val_text = f' val_ema={record["val_acc_ema"]:.4f}' if use_validation else ''
            log(f'Epoch {epoch:03d}/{CFG["epochs"]} lr={lr:.4f} loss={record["train_loss"]:.4f}{val_text} {record["seconds"]}s')

    del loader
    # Evaluation uses the final-epoch averaged weights, so the validation score is not cherry-picked
    final = ema.module
    result = dict(run=name, parameters=PARAMETERS, final_epoch=CFG['epochs'])
    if use_validation:
        result['val_acc'] = accuracy(final, X_train[VAL_IDX], Y_train[VAL_IDX])
        result['val_acc_flip_average'] = accuracy(final, X_train[VAL_IDX], Y_train[VAL_IDX], True)
    result['predicted_class_counts'] = write_submission(final, out / 'submission.csv')
    save_atomic(dict(model=final.state_dict(), widths=CFG['widths'], blocks=CFG['blocks'],
                     parameters=PARAMETERS, mean=MEAN, std=STD, result=result), out / 'final_model.pt')
    (out / 'result.json').write_text(json.dumps(result, indent=2))
    log(json.dumps(result))
    return result

## 8. Run D: train the student on all 50,000 images

This creates `results/distilled_all_data/submission.csv`.

In [9]:
ALL_IDX = np.arange(len(X_train))[:len(TRAIN_IDX)] if SMOKE else np.arange(len(X_train))
results = {'distilled_all_data': run_experiment('distilled_all_data', ALL_IDX, use_validation=False)}


Resumed distilled_all_data after epoch 300
{"run": "distilled_all_data", "parameters": 4980392, "final_epoch": 300, "predicted_class_counts": [177, 221, 202, 205, 209, 213, 178, 197, 197, 201]}


## 9. Curve and summary for the report

In [ ]:
import matplotlib
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = read_rows(ROOT / 'results' / 'distilled_all_data' / 'history.csv')
figure, axis = plt.subplots(figsize=(6, 3.6))
axis.plot([int(row['epoch']) for row in history], [float(row['train_loss']) for row in history])
axis.set(xlabel='Epoch', ylabel='Training loss')
figure.tight_layout()
figure.savefig(ROOT / 'results' / 'distilled_curves.png', dpi=160)
plt.show()

(ROOT / 'results' / 'distilled_summary.json').write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))
print('Submissions:')
for path in sorted((ROOT / 'results').glob('*/submission.csv')):
    print(' ', path)